In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Set random seed
np.random.seed(1234)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

In [ ]:
import os
# Load the preprocessed data

# Set the correct path to your data directory
data_dir = '../data'  # Adjust this path to where your CSV files are located

# Load train, validation, and test sets
X_train = pd.read_csv(os.path.join(data_dir, 'X_train_scaled.csv'))
Y_train = pd.read_csv(os.path.join(data_dir, 'y_train.csv')).squeeze()


X_val = pd.read_csv(os.path.join(data_dir, 'X_val_scaled.csv'))
Y_val = pd.read_csv(os.path.join(data_dir, 'y_val.csv')).squeeze()


X_test = pd.read_csv(os.path.join(data_dir, 'X_test_scaled.csv'))
Y_test = pd.read_csv(os.path.join(data_dir, 'y_test.csv')).squeeze()


print("Data loaded successfully!")
print(f"\nTraining set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"\nTraining labels distribution:\n{Y_train.value_counts()}")


In [ ]:
# NaNs check
print("\nChecking for NaNs in datasets:")
print(f"NaNs in X_train: {X_train.isna().sum().sum()}")
print(f"NaNs in Y_train: {Y_train.isna().sum()}")
print(f"NaNs in X_val: {X_val.isna().sum().sum()}")
print(f"NaNs in Y_val: {Y_val.isna().sum()}")
print(f"NaNs in X_test: {X_test.isna().sum().sum()}")
print(f"NaNs in Y_test: {Y_test.isna().sum()}")

In [ ]:
# Baseline Decision Tree Classifier
from sklearn.tree import DecisionTreeClassifier

# Decision Tree Classifier basic model
dt_clf = DecisionTreeClassifier(
    random_state=1234, 
    max_depth=28, 
    criterion='gini',
    min_samples_split=10)

# fit the model
dt_clf.fit(X_train, Y_train)
# Evaluate on validation set
val_accuracy = dt_clf.score(X_val, Y_val)
print(f"Validation Accuracy of Decision Tree: {val_accuracy:.4f}")

In [ ]:
# Baseline XGBoost model with default hyperparameters (no scale_pos_weight)
xgb_baseline = XGBClassifier(
    random_state=1234,
    n_jobs=-1,
    eval_metric="logloss",
)
# fit the model
xgb_baseline.fit(X_train, Y_train)
# Evaluate on validation set
val_accuracy_xgb = xgb_baseline.score(X_val, Y_val)
print(f"Validation Accuracy of XGBoost Baseline: {val_accuracy_xgb:.4f}")

In [ ]:
# Baseline Logistic Regression model
from sklearn.linear_model import LogisticRegression
logreg_baseline = LogisticRegression(
    random_state=1234,
    class_weight='balanced',
    solver='liblinear',
    n_jobs=-1,
)

# fit the model
logreg_baseline.fit(X_train, Y_train)
# Evaluate on validation set
val_accuracy_logreg = logreg_baseline.score(X_val, Y_val)
print(f"Validation Accuracy of Logistic Regression: {val_accuracy_logreg:.4f}") 

In [ ]:
# Random Foreest Classifier basic model
rf_clf = RandomForestClassifier(
    n_estimators=10,
    max_depth=25,
    bootstrap=True,
    class_weight=None,
    max_leaf_nodes=None,
    criterion='gini',
    min_samples_leaf=1,
    min_samples_split=10,
    min_weight_fraction_leaf=0.0,
    random_state=1234,
    n_jobs=None,
    oob_score=False,
    verbose=0,
    warm_start=False,
)
# fit the model
rf_clf.fit(X_train, Y_train)
# Evaluate on validation set
val_accuracy_rf = rf_clf.score(X_val, Y_val)
print(f"Validation Accuracy of Random Forest: {val_accuracy_rf:.4f}")

In [ ]:
# Compute F1 Scores for all models
from sklearn.metrics import f1_score, classification_report
Y_val_pred = dt_clf.predict(X_val)
val_f1_dt = f1_score(Y_val, Y_val_pred)
print(f"\nValidation F1 Score of Decision Tree: {val_f1_dt:.4f}")
print("Classification Report for Decision Tree:\n", classification_report(Y_val, Y_val_pred))
Y_val_pred_xgb = xgb_baseline.predict(X_val)
val_f1_xgb = f1_score(Y_val, Y_val_pred_xgb)
print(f"\nValidation F1 Score of XGBoost Baseline: {val_f1_xgb:.4f}")
print("Classification Report for XGBoost Baseline:\n", classification_report(Y_val, Y_val_pred_xgb))
Y_val_pred_logreg = logreg_baseline.predict(X_val)
val_f1_logreg = f1_score(Y_val, Y_val_pred_logreg)
print(f"\nValidation F1 Score of Logistic Regression: {val_f1_logreg:.4f}")
print("Classification Report for Logistic Regression:\n", classification_report(Y_val, Y_val_pred_logreg))
Y_val_pred_rf = rf_clf.predict(X_val)
val_f1_rf = f1_score(Y_val, Y_val_pred_rf)      
print(f"\nValidation F1 Score of Random Forest: {val_f1_rf:.4f}")
print("Classification Report for Random Forest:\n", classification_report(Y_val, Y_val_pred_rf))    

In [ ]:
# Applying SMOTE to handle class imbalance
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=1234)
X_train_smote, Y_train_smote = smote.fit_resample(X_train, Y_train)
print(f"\nAfter SMOTE, training set shape: {X_train_smote.shape}")
print(f"After SMOTE, training labels distribution:\n{Y_train_smote.value_counts()}")

In [ ]:
# Retrain all models on SMOTE data and show f1 scores
# Decision Tree
dt_clf.fit(X_train_smote, Y_train_smote)
Y_val_pred_smote_dt = dt_clf.predict(X_val)
val_f1_smote_dt = f1_score(Y_val, Y_val_pred_smote_dt)
print(f"\nAfter SMOTE, Validation F1 Score of Decision Tree: {val_f1_smote_dt:.4f}")
print("Classification Report for Decision Tree after SMOTE:\n", classification_report(Y_val, Y_val_pred_smote_dt))
# XGBoost
xgb_baseline.fit(X_train_smote, Y_train_smote)
Y_val_pred_smote_xgb = xgb_baseline.predict(X_val)
val_f1_smote_xgb = f1_score(Y_val, Y_val_pred_smote_xgb)
print(f"\nAfter SMOTE, Validation F1 Score of XGBoost Baseline: {val_f1_smote_xgb:.4f}")
print("Classification Report for XGBoost Baseline after SMOTE:\n", classification_report(Y_val, Y_val_pred_smote_xgb))
# Logistic Regression
logreg_baseline.fit(X_train_smote, Y_train_smote)
Y_val_pred_smote_logreg = logreg_baseline.predict(X_val)
val_f1_smote_logreg = f1_score(Y_val, Y_val_pred_smote_logreg)
print(f"\nAfter SMOTE, Validation F1 Score of Logistic Regression: {val_f1_smote_logreg:.4f}")
print("Classification Report for Logistic Regression after SMOTE:\n", classification_report(Y_val, Y_val_pred_smote_logreg))
# Random Forest
rf_clf.fit(X_train_smote, Y_train_smote)
Y_val_pred_smote_rf = rf_clf.predict(X_val)
val_f1_smote_rf = f1_score(Y_val, Y_val_pred_smote_rf)      
print(f"\nAfter SMOTE, Validation F1 Score of Random Forest: {val_f1_smote_rf:.4f}")
print("Classification Report for Random Forest after SMOTE:\n", classification_report(Y_val, Y_val_pred_smote_rf))  